# Chapter 18: Euclidean Geometry

**Source orientation.** Perspectives on Projective Geometry, Chapter 18, Sections 18.1-18.8, printed pages 329-348 / PDF pages 351-370. The source is used here only to identify the mathematical route and coverage; the prose, code, diagrams, and checks below are original.

**Chapter goal.** Recover Euclidean ideas - circles, angle, perpendicularity, theorem translation, and distance - from projective data by marking two special complex points on the line at infinity.

**Chapter question.** How can two invisible points, usually written `I=(-i,1,0)^T` and `J=(i,1,0)^T`, let a projective plane remember Euclidean measurement?

The main move in the chapter is not to add rulers and protractors to projective geometry. Instead, we keep projective invariance and add a distinguished pair. Once the pair `{I,J}` is fixed, a conic through both of them behaves like a circle, a transformation fixing or swapping the pair behaves like a similarity, and cross-ratios involving the pair start to measure angle and relative distance.

In [ ]:
from pathlib import Path
import math
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not find the Perspectives on Projective Geometry course root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-18-euclidean-geometry"

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sympy as sp
from PIL import Image

from utils.artifacts import (
    artifact_path,
    assert_artifacts,
    book_relative,
    display_artifact,
    ensure_artifact_root,
    save_json,
    save_table,
)
from utils.conics import circle_conic

ensure_artifact_root(ARTIFACT_ROOT)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 180,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
})

CHAPTER = {
    "number": 18,
    "title": "Euclidean Geometry",
    "source_span": "printed pp. 329-348 / PDF pp. 351-370",
    "sections": "18.1-18.8",
    "artifact_slug": "chapter-18-euclidean-geometry",
}

STORYBOARD = {
    "chapter_goal": "Recover circles, similarities, angle, and distance from projective data plus the circular points I and J.",
    "source_span_read": "Chapter 18, Sections 18.1-18.8, printed pages 329-348 / PDF pages 351-370.",
    "concept_inventory": [
        "circular points I and J on the line at infinity",
        "finite real points transferred to complex coordinates by brackets with I and J",
        "cocircularity as the condition that A,B,C,D,I,J lie on one conic",
        "robust cross-ratio interpretations for four cocircular points",
        "similarity transformations as projective maps preserving the pair {I,J}",
        "Miquel-style theorem translation by replacing cocircularity with six-point conic conditions",
        "orthogonality as harmonic position with I and J",
        "Laguerre's formula for angle from a cross-ratio on the line at infinity",
        "distance as a projective expression normalized by a reference segment",
    ],
    "library_routing_table": [
        {"concept": "symbolic circular-point identities", "library": "SymPy", "why": "exactly checks determinant transfer and circle equations at I and J"},
        {"concept": "static incidence and measurement diagrams", "library": "Matplotlib", "why": "durable labeled PNG figures with equal aspect geometry"},
        {"concept": "transformation and angle parameter exploration", "library": "Plotly", "why": "standalone HTML lets learners inspect families of projective/similarity actions"},
        {"concept": "theorem translation proof state", "library": "NetworkX", "why": "a dependency graph exposes how conic hypotheses feed the final cocircularity claim"},
        {"concept": "invariant summaries", "library": "Pandas", "why": "CSV tables keep numeric checks and lab comparisons inspectable"},
    ],
    "visual_sequence": [
        {"artifact": "figures/circular-points-absolute-conic.png", "inspection_target": "I and J live on the ideal line and satisfy every circle equation", "validation": "symbolic substitution and determinant transfer"},
        {"artifact": "figures/cocircularity-conic-through-IJ.png", "inspection_target": "four real points become cocircular exactly when the six-point conic test with I,J closes", "validation": "bracket residual near zero for circle sample, nonzero for perturbation"},
        {"artifact": "html/robust-cross-ratio-circle.html", "inspection_target": "complex, bracket, and chord-length views of one cross-ratio agree", "validation": "cross-ratio residual below tolerance"},
        {"artifact": "html/euclidean-transformation-family.html", "inspection_target": "similarities fix or swap I,J while shear fails", "validation": "projective equality checks on transformed circular points"},
        {"artifact": "figures/miquel-projective-translation-graph.png", "inspection_target": "Miquel hypotheses become conic dependencies with I,J", "validation": "graph path to conclusion and recorded bracket dependencies"},
        {"artifact": "figures/orthogonality-harmonic-range.png", "inspection_target": "right angle means the ideal directions and I,J have cross-ratio -1", "validation": "harmonic cross-ratio residual"},
        {"artifact": "html/laguerre-angle-recovery.html", "inspection_target": "angle is recovered from a cross-ratio on the line at infinity", "validation": "Laguerre residual across a grid of directions"},
        {"artifact": "figures/distance-from-projective-data.png", "inspection_target": "distance ratio comes from brackets with I,J plus a unit reference", "validation": "projective formula matches Euclidean distances"},
    ],
    "artifact_plan": {
        "figures": [
            "circular-points-absolute-conic.png",
            "cocircularity-conic-through-IJ.png",
            "miquel-projective-translation-graph.png",
            "orthogonality-harmonic-range.png",
            "laguerre-angle-recovery.png",
            "distance-from-projective-data.png",
        ],
        "html": ["robust-cross-ratio-circle.html", "euclidean-transformation-family.html", "laguerre-angle-recovery.html"],
        "tables": ["cross-ratio-robustness.csv", "distance-formula-samples.csv", "projective-measurement-lab.csv"],
        "checks": ["storyboard.json", "visual-checks.json", "final-sanity.json"],
    },
    "computational_checks": [
        "I and J lie on the line at infinity and on every circle in the standard embedding",
        "bracket transfer [p,I,l_inf]=x+iy and [p,J,l_inf]=x-iy",
        "cocircular bracket residual and perturbed counterexample residual",
        "similarity matrices preserve or swap {I,J}; shear does not",
        "orthogonal lines produce cross-ratio -1 with I,J",
        "Laguerre angle recovery agrees modulo pi",
        "projective distance formula agrees with Euclidean distance after unit normalization",
    ],
    "proof_visualization_strategy": "Use a NetworkX dependency graph for the Miquel translation: each cocircular hypothesis becomes a conic through I,J; bracket cancellation feeds the final E,F,G,H conic condition.",
    "risks": [
        "I and J are complex, so diagrams must mark them schematically rather than as ordinary real points",
        "Laguerre angles are naturally modulo pi, so numeric checks compare in that quotient",
        "distance needs a unit reference segment because absolute length is not a similarity invariant",
    ],
    "acceptance_criteria": [
        "notebook executes with nbclient or validate_ppg_course.py --chapter 18",
        "all artifacts live under artifacts/chapter-18-euclidean-geometry",
        "visual-checks.json reports all_files_exist and cross_ratio_error <= 1e-9",
        "final-sanity.json records concept checks and nonzero artifact sizes",
    ],
}
storyboard_path = save_json(STORYBOARD, ARTIFACT_ROOT, "checks", "storyboard.json")
book_relative(storyboard_path)

## Computational Translation Guide

The chapter works with two coordinate worlds at once.

| Book object | Computational representation here | What is checked |
| --- | --- | --- |
| Finite Euclidean point `(x,y)` | homogeneous vector `(x,y,1)^T` | bracket transfer to `x+iy` and `x-iy` |
| Circular points | `I=(-i,1,0)^T`, `J=(i,1,0)^T` in `CP^2` | both lie on `l_infinity` and on every circle equation |
| Circle | real conic passing through `I` and `J` | six-point conic/cocircularity residual |
| Similarity | real projective matrix preserving or swapping `{I,J}` | projective equality of transformed circular points |
| Right angle | harmonic range with `I,J` on the ideal line | cross-ratio equals `-1` |
| General angle | Laguerre formula `(1/(2i)) log(M,L;I,J)` | recovered angle modulo `pi` |
| Distance | bracket expression normalized by a unit segment | projective formula equals Euclidean length |

The notebook uses SymPy where exact algebra matters, Matplotlib for durable 2D construction diagrams, Plotly for families that are better inspected interactively, NetworkX for proof state, and Pandas for tables of numerical invariants.

In [ ]:
I = np.array([-1j, 1 + 0j, 0 + 0j], dtype=complex)
J = np.array([1j, 1 + 0j, 0 + 0j], dtype=complex)
L_INF = np.array([0 + 0j, 0 + 0j, 1 + 0j], dtype=complex)


def hp(x, y, w=1):
    return np.array([complex(x), complex(y), complex(w)], dtype=complex)


def bracket3(a, b, c):
    return np.linalg.det(np.column_stack([a, b, c]))


def complex_coord(p):
    return bracket3(p, I, L_INF)


def conjugate_coord(p):
    return bracket3(p, J, L_INF)


def cross_ratio_complex(a, b, c, d):
    return ((a - c) * (b - d)) / ((a - d) * (b - c))


def cocircular_bracket_residual(A, B, C, D):
    lhs = bracket3(A, C, I) * bracket3(B, D, I) * bracket3(A, D, J) * bracket3(B, C, J)
    rhs = bracket3(A, C, J) * bracket3(B, D, J) * bracket3(A, D, I) * bracket3(B, C, I)
    return float(abs(lhs - rhs) / max(1.0, abs(lhs), abs(rhs)))


def projectively_equal(p, q, tol=1e-9):
    p = np.asarray(p, dtype=complex)
    q = np.asarray(q, dtype=complex)
    return bool(np.linalg.norm(np.cross(p, q)) <= tol * max(1.0, np.linalg.norm(p) * np.linalg.norm(q)))


def line_from_angle(theta, offset=0.0):
    return np.array([-math.sin(theta), math.cos(theta), offset], dtype=complex)


def infinity_point_of_line(line):
    return np.cross(np.asarray(line, dtype=complex), L_INF)


def cross_ratio_at_infinity(line_l, line_m):
    L = infinity_point_of_line(line_l)
    M = infinity_point_of_line(line_m)
    return (bracket3(M, I, L_INF) * bracket3(L, J, L_INF)) / (bracket3(M, J, L_INF) * bracket3(L, I, L_INF))


def laguerre_angle_mod_pi(line_l, line_m):
    cr = cross_ratio_at_infinity(line_l, line_m)
    return float((0.5 * np.angle(cr)) % np.pi)


def mod_pi_error(a, b):
    return float(abs(((a - b + np.pi / 2) % np.pi) - np.pi / 2))


def projective_distance(P, Q, A_ref, B_ref):
    numerator = bracket3(P, Q, I) * bracket3(P, Q, J) * bracket3(A_ref, I, J) * bracket3(B_ref, I, J)
    denominator = bracket3(A_ref, B_ref, I) * bracket3(A_ref, B_ref, J) * bracket3(P, I, J) * bracket3(Q, I, J)
    value = np.sqrt(numerator / denominator)
    value = np.real_if_close(value, tol=1000)
    return float(abs(value))


def affine_xy(p):
    p = np.asarray(p, dtype=complex)
    return np.array([p[0] / p[2], p[1] / p[2]], dtype=complex)


def finite_transform(T, points):
    pts = np.column_stack(points)
    out = (T @ pts).T
    return [row / row[2] for row in out]


def save_figure(fig, category, filename):
    path = artifact_path(ARTIFACT_ROOT, category, filename)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def artifact_size_map(paths):
    return {book_relative(path): int(Path(path).stat().st_size) for path in paths}


def png_stats(path):
    image = Image.open(path).convert("RGB")
    arr = np.asarray(image, dtype=float)
    return {
        "path": book_relative(path),
        "width": int(image.width),
        "height": int(image.height),
        "pixel_std": float(arr.std()),
        "file_size": int(Path(path).stat().st_size),
    }


x, y, z, a, b, c = sp.symbols("x y z a b c", real=True)
poly = x**2 + y**2 + a*x*z + b*y*z + c*z**2
assert sp.simplify(poly.subs({x: -sp.I, y: 1, z: 0})) == 0
assert sp.simplify(poly.subs({x: sp.I, y: 1, z: 0})) == 0

px, py = sp.symbols("px py", real=True)
P_sym = sp.Matrix([px, py, 1])
I_sym = sp.Matrix([-sp.I, 1, 0])
J_sym = sp.Matrix([sp.I, 1, 0])
L_sym = sp.Matrix([0, 0, 1])
br_PI = sp.Matrix.hstack(P_sym, I_sym, L_sym).det()
br_PJ = sp.Matrix.hstack(P_sym, J_sym, L_sym).det()
assert sp.simplify(br_PI - (px + sp.I*py)) == 0
assert sp.simplify(br_PJ - (px - sp.I*py)) == 0

core_checks = {
    "I_on_line_at_infinity": bool(abs(I[2]) < 1e-12),
    "J_on_line_at_infinity": bool(abs(J[2]) < 1e-12),
    "I_cross_J_spans_l_infinity": projectively_equal(np.cross(I, J), L_INF),
    "symbolic_circle_substitution_I": "0",
    "symbolic_circle_substitution_J": "0",
    "bracket_transfer_I": str(br_PI),
    "bracket_transfer_J": str(br_PJ),
}
core_checks_path = save_json(core_checks, ARTIFACT_ROOT, "checks", "circular-points-symbolic.json")
core_checks

## 18.1 Circular Points As Euclidean Anchors

`I` and `J` are not ordinary real points. They are complex points at infinity. Their job is to mark the two ideal directions that every real circle shares after projective completion.

A circle in the standard affine chart has homogeneous equation

\[
x^2+y^2+a xz+b yz+c z^2=0.
\]

Substituting `I=(-i,1,0)^T` or `J=(i,1,0)^T` kills the equation because `(-i)^2+1=0` and `i^2+1=0`. The figure below keeps the real circle visible while showing the ideal line as a separate schematic band. Inspect the band: the two circular points are fixed anchors, not sampled points on the real curve.

![Circular points as absolute conic anchors](../../artifacts/chapter-18-euclidean-geometry/figures/circular-points-absolute-conic.png)

In [ ]:
theta = np.linspace(0, 2*np.pi, 400)
centers = [(-0.8, 0.0), (0.7, 0.35), (0.2, -0.55)]
radii = [0.75, 0.55, 0.95]
colors = ["#3366cc", "#c43c39", "#2a9d55"]

fig, (ax, ideal) = plt.subplots(1, 2, figsize=(10.5, 4.1), gridspec_kw={"width_ratios": [3.1, 1.4]})
for (cx, cy), r, color in zip(centers, radii, colors):
    ax.plot(cx + r*np.cos(theta), cy + r*np.sin(theta), color=color, lw=2)
    ax.scatter([cx], [cy], color=color, s=18)
    ax.text(cx + 0.03, cy + 0.04, f"center ({cx:.1f},{cy:.1f})", color=color, fontsize=8)
ax.axhline(0, color="#777777", lw=0.7, alpha=0.5)
ax.axvline(0, color="#777777", lw=0.7, alpha=0.5)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.9, 1.9)
ax.set_ylim(-1.65, 1.65)
ax.set_title("Real circles in the affine chart z=1")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(alpha=0.18)

ideal.plot([0, 1], [0.55, 0.55], color="#111111", lw=2)
ideal.text(0.5, 0.64, "line at infinity z=0", ha="center", fontsize=9)
ideal.scatter([0.24, 0.76], [0.55, 0.55], s=85, color=["#7b3294", "#008080"], zorder=3)
ideal.text(0.24, 0.39, "I=(-i,1,0)", ha="center", color="#7b3294", fontsize=9)
ideal.text(0.76, 0.39, "J=(i,1,0)", ha="center", color="#008080", fontsize=9)
ideal.text(0.5, 0.16, "Every real circle\npasses through both\ncomplex anchors.", ha="center", fontsize=9)
ideal.set_xlim(0, 1)
ideal.set_ylim(0, 1)
ideal.axis("off")
fig.suptitle("Circular points turn Euclidean circles into projective conics", y=1.02)

circular_points_png = save_figure(fig, "figures", "circular-points-absolute-conic.png")
display_artifact(circular_points_png, width=820)

circular_numeric_checks = {
    "circle_values_at_IJ": [],
    "line_at_infinity_values": {"I_z": float(abs(I[2])), "J_z": float(abs(J[2]))},
}
for (cx, cy), r in zip(centers, radii):
    Cmat = circle_conic(cx, cy, r).astype(complex)
    circular_numeric_checks["circle_values_at_IJ"].append({
        "center": [cx, cy],
        "radius": r,
        "value_at_I_abs": float(abs(I @ Cmat @ I)),
        "value_at_J_abs": float(abs(J @ Cmat @ J)),
    })
circular_points_checks_path = save_json(circular_numeric_checks, ARTIFACT_ROOT, "checks", "circular-points-checks.json")
circular_numeric_checks

## 18.2 Cocircularity Becomes A Six-Point Conic Test

The chapter's first major translation is:

\[
A,B,C,D \text{ are cocircular}
\quad\Longleftrightarrow\quad
A,B,C,D,I,J \text{ lie on one conic}.
\]

Computationally, the condition can be evaluated by the bracket identity

\[
[ACI][BDI][ADJ][BCJ]=[ACJ][BDJ][ADI][BCI].
\]

The figure compares four points on a true circle with the same data after one point is moved. Inspect the residual table: the real drawing alone is not the proof; the bracket residual is the projective certificate.

![Cocircularity bracket test](../../artifacts/chapter-18-euclidean-geometry/figures/cocircularity-conic-through-IJ.png)

In [ ]:
circle_center = np.array([0.25, -0.10])
circle_radius = 1.25
angles = np.deg2rad([25, 112, 205, 315])
A, B, C, D = [hp(*(circle_center + circle_radius*np.array([np.cos(t), np.sin(t)]))) for t in angles]
D_bad = hp(*(circle_center + np.array([1.55*np.cos(angles[3]), 0.72*np.sin(angles[3])])))

samples = [
    {"case": "four points on one circle", "A": A, "B": B, "C": C, "D": D},
    {"case": "D moved off the circle", "A": A, "B": B, "C": C, "D": D_bad},
]
rows = []
for item in samples:
    residual = cocircular_bracket_residual(item["A"], item["B"], item["C"], item["D"])
    z_vals = [complex_coord(item[key]) for key in ["A", "B", "C", "D"]]
    cr = cross_ratio_complex(*z_vals)
    rows.append({
        "case": item["case"],
        "bracket_residual": residual,
        "cross_ratio_real_part": float(cr.real),
        "cross_ratio_imag_abs": float(abs(cr.imag)),
        "circle_center_x": float(circle_center[0]),
        "circle_center_y": float(circle_center[1]),
        "circle_radius": float(circle_radius),
        "inspection_target": "zero residual means A,B,C,D,I,J pass the conic test",
        "failure_mode": "moving one real point breaks reality of the cross-ratio and the six-point conic identity",
    })

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3), sharex=True, sharey=True)
labels = ["A", "B", "C", "D"]
for ax, item, title in zip(axes, samples, ["Cocircular", "Perturbed"]):
    ax.plot(circle_center[0] + circle_radius*np.cos(theta), circle_center[1] + circle_radius*np.sin(theta), color="#555555", lw=1.5, alpha=0.65)
    pts = [affine_xy(item[k]).real for k in labels]
    xy = np.array(pts)
    ax.scatter(xy[:,0], xy[:,1], color=["#1b4f9c", "#d1495b", "#2a9d55", "#f4a261"], s=46, zorder=3)
    for label, (xx, yy) in zip(labels, xy):
        ax.text(xx + 0.04, yy + 0.04, label, fontsize=10)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"{title}\nresidual={cocircular_bracket_residual(item['A'], item['B'], item['C'], item['D']):.2e}")
    ax.grid(alpha=0.18)
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
fig.suptitle("Cocircularity is tested by adding I and J to the conic data", y=1.02)

cocircular_png = save_figure(fig, "figures", "cocircularity-conic-through-IJ.png")
cocircular_csv = save_table(rows, ARTIFACT_ROOT, "tables", "cocircularity-bracket-residuals.csv")
cocircular_checks_path = save_json({"cases": rows}, ARTIFACT_ROOT, "checks", "cocircularity-checks.json")
display_artifact(cocircular_png, width=820)
display_artifact(cocircular_csv)

cocircular_good_residual = rows[0]["bracket_residual"]
cocircular_bad_residual = rows[1]["bracket_residual"]
rows

## 18.3 Robust Cross-Ratios On A Circle

For four cocircular points, the same projective invariant can be read in several ways:

- as a complex cross-ratio of the corresponding numbers in `CP^1`,
- as a bracket expression in `RP^2` with `I`,
- as a real quantity whose magnitude is a ratio of chord lengths.

The interactive artifact below lets the circle geometry and the algebra sit in the same frame. The important inspection target is the table: the imaginary part of the cross-ratio is numerical noise, and the magnitude agrees with the chord-length expression.

[Open the robust cross-ratio HTML artifact](../../artifacts/chapter-18-euclidean-geometry/html/robust-cross-ratio-circle.html)

In [ ]:
def circle_point(center, radius, deg):
    t = math.radians(deg)
    return hp(center[0] + radius*math.cos(t), center[1] + radius*math.sin(t))

robust_angles = [10, 75, 165, 285]
P_A, P_B, P_C, P_D = [circle_point((0.0, 0.0), 1.0, deg) for deg in robust_angles]
zA, zB, zC, zD = [complex_coord(p) for p in [P_A, P_B, P_C, P_D]]
cr_complex = cross_ratio_complex(zA, zB, zC, zD)
cr_bracket = (bracket3(P_A, P_C, I) * bracket3(P_B, P_D, I)) / (bracket3(P_A, P_D, I) * bracket3(P_B, P_C, I))
chord_ratio = abs(zA - zC) * abs(zB - zD) / (abs(zA - zD) * abs(zB - zC))

cross_rows = [
    {"method": "complex cross-ratio", "real_part": float(cr_complex.real), "imag_abs": float(abs(cr_complex.imag)), "magnitude": float(abs(cr_complex))},
    {"method": "bracket expression with I", "real_part": float(cr_bracket.real), "imag_abs": float(abs(cr_bracket.imag)), "magnitude": float(abs(cr_bracket))},
    {"method": "chord-length magnitude", "real_part": float(chord_ratio), "imag_abs": 0.0, "magnitude": float(chord_ratio)},
]
cross_ratio_error = float(max(abs(cr_complex - cr_bracket), abs(abs(cr_complex) - chord_ratio), abs(cr_complex.imag)))

unit = np.exp(1j*np.linspace(0, 2*np.pi, 360))
fig = go.Figure()
fig.add_trace(go.Scatter(x=unit.real, y=unit.imag, mode="lines", name="circle", line=dict(color="#444444")))
point_names = ["A", "B", "C", "D"]
zs = [zA, zB, zC, zD]
fig.add_trace(go.Scatter(
    x=[z.real for z in zs],
    y=[z.imag for z in zs],
    mode="markers+text",
    text=point_names,
    textposition="top center",
    marker=dict(size=11, color=["#1b4f9c", "#d1495b", "#2a9d55", "#f4a261"]),
    name="sample points",
))
for name, z1, z2, color in [
    ("|A-C|", zA, zC, "#1b4f9c"),
    ("|B-D|", zB, zD, "#d1495b"),
    ("|A-D|", zA, zD, "#777777"),
    ("|B-C|", zB, zC, "#777777"),
]:
    fig.add_trace(go.Scatter(x=[z1.real, z2.real], y=[z1.imag, z2.imag], mode="lines", name=name, line=dict(color=color, dash="dot")))
fig.update_layout(
    title="Robust cross-ratio readings for four cocircular points",
    xaxis=dict(scaleanchor="y", scaleratio=1, title="real part"),
    yaxis=dict(title="imaginary part"),
    width=780,
    height=560,
    margin=dict(l=40, r=20, t=60, b=40),
)
robust_html = artifact_path(ARTIFACT_ROOT, "html", "robust-cross-ratio-circle.html")
fig.write_html(robust_html, include_plotlyjs=True, full_html=True)
robust_csv = save_table(cross_rows, ARTIFACT_ROOT, "tables", "cross-ratio-robustness.csv")
robust_checks_path = save_json({"cross_ratio_error": cross_ratio_error, "rows": cross_rows}, ARTIFACT_ROOT, "checks", "cross-ratio-robustness.json")

display_artifact(robust_html, height=520)
display_artifact(robust_csv)
cross_ratio_error

## 18.4 Similarities Are The Maps That Preserve The Pair `{I,J}`

In ordinary coordinates, an orientation-preserving similarity has the form

\[
\begin{pmatrix} c&s&a\\ -s&c&b\\ 0&0&1\end{pmatrix},
\]

while an orientation-reversing one has the form

\[
\begin{pmatrix} c&s&a\\ s&-c&b\\ 0&0&1\end{pmatrix}.
\]

The projective characterization is cleaner: the first type fixes both circular points projectively; the second swaps them. A shear may preserve the line at infinity, but it does not preserve the pair `{I,J}`, so it is affine rather than similarity geometry. Inspect the HTML panels: the similarity sends a circle to a circle, the reflection also sends a circle to a circle, and the shear turns the circle into an ellipse.

In [ ]:
def similarity_preserving(scale, angle, tx, ty):
    c = scale * math.cos(angle)
    s = scale * math.sin(angle)
    return np.array([[c, s, tx], [-s, c, ty], [0, 0, 1]], dtype=complex)


def similarity_reversing(scale, angle, tx, ty):
    c = scale * math.cos(angle)
    s = scale * math.sin(angle)
    return np.array([[c, s, tx], [s, -c, ty], [0, 0, 1]], dtype=complex)


def shear(k):
    return np.array([[1, k, 0], [0, 1, 0], [0, 0, 1]], dtype=complex)

S = similarity_preserving(0.85, math.radians(28), 0.35, -0.20)
R = similarity_reversing(0.85, math.radians(28), 0.35, -0.20)
H = shear(0.65)
transformations = [("orientation-preserving similarity", S), ("orientation-reversing similarity", R), ("affine shear", H)]
transformation_checks = []
for name, T in transformations:
    TI = T @ I
    TJ = T @ J
    transformation_checks.append({
        "transformation": name,
        "maps_I_to_I": projectively_equal(TI, I),
        "maps_J_to_J": projectively_equal(TJ, J),
        "maps_I_to_J": projectively_equal(TI, J),
        "maps_J_to_I": projectively_equal(TJ, I),
        "preserves_pair": (projectively_equal(TI, I) and projectively_equal(TJ, J)) or (projectively_equal(TI, J) and projectively_equal(TJ, I)),
    })

circle_pts = [hp(math.cos(t), math.sin(t)) for t in np.linspace(0, 2*np.pi, 180)]
fig = make_subplots(rows=1, cols=3, subplot_titles=[item[0] for item in transformations])
for col, (name, T) in enumerate(transformations, start=1):
    out = finite_transform(T, circle_pts)
    xy = np.array([[p[0].real, p[1].real] for p in out])
    fig.add_trace(go.Scatter(x=xy[:,0], y=xy[:,1], mode="lines", name=name, showlegend=False, line=dict(width=3)), row=1, col=col)
    fig.update_xaxes(scaleanchor=f"y{col}" if col > 1 else "y", scaleratio=1, row=1, col=col)
    fig.update_xaxes(range=[-1.6, 1.8], row=1, col=col)
    fig.update_yaxes(range=[-1.7, 1.5], row=1, col=col)
fig.update_layout(title="Which projective maps preserve the circular pair?", width=1050, height=420, margin=dict(l=40, r=20, t=70, b=40))
transform_html = artifact_path(ARTIFACT_ROOT, "html", "euclidean-transformation-family.html")
fig.write_html(transform_html, include_plotlyjs=True, full_html=True)
transform_checks_path = save_json({"transformations": transformation_checks}, ARTIFACT_ROOT, "checks", "transformation-fixed-circular-points.json")

display_artifact(transform_html, height=430)
transformation_checks

## 18.5 Translating Theorems: Miquel As Projective Bookkeeping

The chapter's theorem-translation pattern is powerful because it turns a Euclidean-sounding statement into a projective dependency chain.

For Miquel-style data, each hypothesis that four real points are cocircular becomes a statement that six points lie on a conic after `I` and `J` are added. The proof then uses bracket products and cancellation to force the final four points onto a conic with `I` and `J`.

The graph is intentionally a proof-state artifact rather than a decorative theorem picture. Inspect which nodes touch `I,J`: the circular points are the common projective payload carried by every cocircularity hypothesis.

![Miquel theorem translation graph](../../artifacts/chapter-18-euclidean-geometry/figures/miquel-projective-translation-graph.png)

In [ ]:
G = nx.DiGraph()
positions = {
    "absolute pair I,J": (0.0, 0.0),
    "ABCDIJ conic": (-2.1, 1.2),
    "ABEFIJ conic": (-1.05, 1.8),
    "BCFGIJ conic": (0.0, 2.05),
    "CDGHIJ conic": (1.05, 1.8),
    "DAHEIJ conic": (2.1, 1.2),
    "bracket product\nand cancellation": (0.0, 0.95),
    "EFGHIJ conic": (0.0, -1.05),
    "E,F,G,H\ncocircular": (0.0, -2.0),
}
for node in positions:
    G.add_node(node)
for node in ["ABCDIJ conic", "ABEFIJ conic", "BCFGIJ conic", "CDGHIJ conic", "DAHEIJ conic"]:
    G.add_edge("absolute pair I,J", node)
    G.add_edge(node, "bracket product\nand cancellation")
G.add_edge("bracket product\nand cancellation", "EFGHIJ conic")
G.add_edge("absolute pair I,J", "EFGHIJ conic")
G.add_edge("EFGHIJ conic", "E,F,G,H\ncocircular")

fig, ax = plt.subplots(figsize=(9.2, 6.0))
node_colors = []
for node in G.nodes:
    if node == "absolute pair I,J":
        node_colors.append("#7b3294")
    elif "conic" in node:
        node_colors.append("#80cdc1")
    elif "bracket" in node:
        node_colors.append("#f4a261")
    else:
        node_colors.append("#c7eae5")
nx.draw_networkx_edges(G, positions, ax=ax, arrowstyle="-|>", arrowsize=14, edge_color="#555555", width=1.4)
nx.draw_networkx_nodes(G, positions, ax=ax, node_color=node_colors, node_size=2100, edgecolors="#222222", linewidths=0.8)
nx.draw_networkx_labels(G, positions, ax=ax, font_size=8)
ax.set_title("Projective proof state for theorem translation")
ax.axis("off")
miquel_png = save_figure(fig, "figures", "miquel-projective-translation-graph.png")
miquel_data = {
    "nodes": list(G.nodes),
    "edges": [[u, v] for u, v in G.edges],
    "has_path_to_conclusion": nx.has_path(G, "absolute pair I,J", "E,F,G,H\ncocircular"),
    "hypothesis_conic_count": 5,
}
miquel_checks_path = save_json(miquel_data, ARTIFACT_ROOT, "checks", "miquel-translation-dependencies.json")
display_artifact(miquel_png, width=780)
miquel_data

## 18.6 Orthogonality Is Harmonic Position At Infinity

Two finite lines meet the line at infinity in two ideal direction points `L` and `M`. The chapter's orthogonality criterion says that the two pairs `{L,M}` and `{I,J}` are in harmonic position. In cross-ratio form, that means

\[
(M,L;I,J)=-1.
\]

This is the right-angle special case of Laguerre's formula. Inspect both panels: the left panel shows ordinary perpendicular lines; the right panel shows the actual projective test happening on the ideal line.

![Orthogonality as a harmonic range](../../artifacts/chapter-18-euclidean-geometry/figures/orthogonality-harmonic-range.png)

In [ ]:
theta_l = math.radians(22)
theta_m = theta_l + math.pi / 2
line_l = line_from_angle(theta_l, 0.0)
line_m = line_from_angle(theta_m, 0.0)
orth_cr = cross_ratio_at_infinity(line_l, line_m)
orthogonality_residual = float(abs(orth_cr + 1))

xline = np.linspace(-1.2, 1.2, 50)
def line_points(theta_value):
    return np.column_stack([xline*math.cos(theta_value), xline*math.sin(theta_value)])

fig, (ax, ideal) = plt.subplots(1, 2, figsize=(10.5, 4.2), gridspec_kw={"width_ratios": [2.5, 1.6]})
for theta_value, color, label in [(theta_l, "#1b4f9c", "l"), (theta_m, "#d1495b", "m")]:
    pts = line_points(theta_value)
    ax.plot(pts[:,0], pts[:,1], lw=2.5, color=color, label=label)
ax.scatter([0], [0], color="#222222", s=30, zorder=4)
arc_t = np.linspace(theta_l, theta_m, 80)
ax.plot(0.32*np.cos(arc_t), 0.32*np.sin(arc_t), color="#333333", lw=1.2)
ax.text(0.18, 0.28, "90 deg", fontsize=9)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.grid(alpha=0.18)
ax.legend(loc="upper right")
ax.set_title("Finite view")

ideal.plot([0.05, 0.95], [0.55, 0.55], color="#111111", lw=2)
ideal.scatter([0.22, 0.41, 0.59, 0.78], [0.55]*4, s=[55, 85, 85, 55], color=["#1b4f9c", "#7b3294", "#008080", "#d1495b"], zorder=3)
for x_pos, label, color in [(0.22, "L", "#1b4f9c"), (0.41, "I", "#7b3294"), (0.59, "J", "#008080"), (0.78, "M", "#d1495b")]:
    ideal.text(x_pos, 0.40, label, ha="center", color=color, fontsize=11)
ideal.text(0.5, 0.74, "ideal line", ha="center", fontsize=9)
ideal.text(0.5, 0.12, f"(M,L;I,J) = {orth_cr.real:.1f}{orth_cr.imag:+.1e}i", ha="center", fontsize=9)
ideal.set_xlim(0, 1)
ideal.set_ylim(0, 1)
ideal.axis("off")
ideal.set_title("Projective test")
fig.suptitle("Orthogonality becomes a harmonic range with the circular points", y=1.02)

orthogonality_png = save_figure(fig, "figures", "orthogonality-harmonic-range.png")
orthogonality_checks_path = save_json({
    "cross_ratio_real": float(orth_cr.real),
    "cross_ratio_imag_abs": float(abs(orth_cr.imag)),
    "orthogonality_residual_abs_cr_plus_one": orthogonality_residual,
}, ARTIFACT_ROOT, "checks", "angle-recovery-orthogonality.json")
display_artifact(orthogonality_png, width=820)
orthogonality_residual

## 18.7 Laguerre's Formula Recovers Angle From A Cross-Ratio

Let `l` and `m` be finite lines, and let `L` and `M` be their intersections with the line at infinity. Laguerre's formula reads

\[
\angle(l,m) = {1\over 2i}\log (M,L;I,J) \pmod{\pi}.
\]

The logarithm is multivalued, and unoriented lines are measured modulo `pi`, so the validation below compares angles in that quotient. Inspect the static and interactive artifacts: the recovered curve tracks the true direction difference except for the expected branch behavior, which disappears after comparing modulo `pi`.

![Laguerre angle recovery](../../artifacts/chapter-18-euclidean-geometry/figures/laguerre-angle-recovery.png)

[Open the Laguerre angle HTML artifact](../../artifacts/chapter-18-euclidean-geometry/html/laguerre-angle-recovery.html)

In [ ]:
base_theta = math.radians(17)
phis = np.linspace(base_theta, base_theta + np.pi, 121)
laguerre_rows = []
for phi in phis:
    l = line_from_angle(base_theta, 0.0)
    m = line_from_angle(float(phi), 0.0)
    cr = cross_ratio_at_infinity(l, m)
    recovered = laguerre_angle_mod_pi(l, m)
    expected = float((phi - base_theta) % np.pi)
    laguerre_rows.append({
        "line_l_degrees": math.degrees(base_theta),
        "line_m_degrees": math.degrees(phi),
        "expected_angle_degrees": math.degrees(expected),
        "laguerre_angle_degrees": math.degrees(recovered),
        "mod_pi_error": mod_pi_error(recovered, expected),
        "cross_ratio_real": float(cr.real),
        "cross_ratio_imag": float(cr.imag),
        "cross_ratio_abs_minus_one": float(abs(abs(cr) - 1)),
    })
laguerre_max_error = float(max(row["mod_pi_error"] for row in laguerre_rows))

fig, ax = plt.subplots(figsize=(7.6, 4.4))
ax.plot([row["expected_angle_degrees"] for row in laguerre_rows], [row["laguerre_angle_degrees"] for row in laguerre_rows], color="#1b4f9c", lw=2.5, label="Laguerre recovered")
ax.plot([0, 180], [0, 180], color="#555555", ls="--", lw=1, label="expected modulo pi")
ax.set_xlabel("expected angle (degrees, modulo pi)")
ax.set_ylabel("recovered angle (degrees)")
ax.set_title("Laguerre formula recovers line angle from ideal cross-ratio")
ax.grid(alpha=0.22)
ax.legend()
laguerre_png = save_figure(fig, "figures", "laguerre-angle-recovery.png")

fig_html = go.Figure()
fig_html.add_trace(go.Scatter(
    x=[row["expected_angle_degrees"] for row in laguerre_rows],
    y=[row["laguerre_angle_degrees"] for row in laguerre_rows],
    mode="lines",
    name="Laguerre recovered",
    line=dict(color="#1b4f9c", width=3),
))
fig_html.add_trace(go.Scatter(x=[0, 180], y=[0, 180], mode="lines", name="expected", line=dict(color="#555555", dash="dash")))
fig_html.update_layout(
    title="Laguerre angle recovery modulo pi",
    xaxis_title="expected angle (degrees)",
    yaxis_title="recovered angle (degrees)",
    width=780,
    height=500,
)
laguerre_html = artifact_path(ARTIFACT_ROOT, "html", "laguerre-angle-recovery.html")
fig_html.write_html(laguerre_html, include_plotlyjs=True, full_html=True)
laguerre_csv = save_table(laguerre_rows, ARTIFACT_ROOT, "tables", "laguerre-angle-recovery.csv")
laguerre_checks_path = save_json({
    "max_mod_pi_error": laguerre_max_error,
    "max_cross_ratio_abs_minus_one": float(max(row["cross_ratio_abs_minus_one"] for row in laguerre_rows)),
    "sample_count": len(laguerre_rows),
}, ARTIFACT_ROOT, "checks", "laguerre-angle-recovery.json")

display_artifact(laguerre_png, width=760)
display_artifact(laguerre_html, height=500)
laguerre_max_error

## 18.8 Distance From Projective Data Needs A Reference Segment

The pair `{I,J}` is enough for similarity geometry, but absolute distance is not preserved by similarities. The chapter therefore normalizes distance by a reference segment `AB`. With `|AB|=1`, the projective expression is

\[
|P,Q| = \sqrt{\frac{[P,Q,I][P,Q,J][A,I,J][B,I,J]}{[A,B,I][A,B,J][P,I,J][Q,I,J]}}.
\]

The square root records the unavoidable sign choice; the positive branch gives the ordinary Euclidean length in the standard chart. Inspect the segment samples: all finite points are ordinary real points, but every distance comparison still passes through `I` and `J`.

![Distance recovered from projective brackets](../../artifacts/chapter-18-euclidean-geometry/figures/distance-from-projective-data.png)

In [ ]:
A_ref = hp(0, 0)
B_ref = hp(1, 0)
segments = [
    ("PQ", hp(-0.65, 0.30), hp(0.75, 0.95)),
    ("RS", hp(-0.85, -0.70), hp(0.15, -0.15)),
    ("TU", hp(0.35, -0.95), hp(1.15, 0.25)),
    ("VW", hp(-0.20, 0.85), hp(0.95, -0.55)),
]
distance_rows = []
for name, P, Q in segments:
    Pxy = affine_xy(P).real
    Qxy = affine_xy(Q).real
    euclidean = float(np.linalg.norm(Pxy - Qxy))
    projective = projective_distance(P, Q, A_ref, B_ref)
    distance_rows.append({
        "segment": name,
        "euclidean_distance": euclidean,
        "projective_distance": projective,
        "absolute_error": abs(euclidean - projective),
    })
distance_max_error = float(max(row["absolute_error"] for row in distance_rows))

fig, ax = plt.subplots(figsize=(7.3, 5.1))
ax.plot([0, 1], [0, 0], color="#222222", lw=3, label="unit reference AB")
ax.scatter([0, 1], [0, 0], color="#222222", s=42)
ax.text(0, 0.06, "A", ha="center")
ax.text(1, 0.06, "B", ha="center")
colors = ["#1b4f9c", "#d1495b", "#2a9d55", "#f4a261"]
for (name, P, Q), color, row in zip(segments, colors, distance_rows):
    Pxy = affine_xy(P).real
    Qxy = affine_xy(Q).real
    ax.plot([Pxy[0], Qxy[0]], [Pxy[1], Qxy[1]], color=color, lw=2.5)
    ax.scatter([Pxy[0], Qxy[0]], [Pxy[1], Qxy[1]], color=color, s=36)
    mid = (Pxy + Qxy) / 2
    ax.text(mid[0], mid[1] + 0.06, f"{name}: {row['projective_distance']:.3f}", color=color, ha="center", fontsize=9)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.05, 1.35)
ax.set_ylim(-1.1, 1.15)
ax.grid(alpha=0.18)
ax.set_title("Projective distance formula with unit reference segment")
ax.legend(loc="lower right")
distance_png = save_figure(fig, "figures", "distance-from-projective-data.png")
distance_csv = save_table(distance_rows, ARTIFACT_ROOT, "tables", "distance-formula-samples.csv")
distance_checks_path = save_json({
    "reference_segment_length": projective_distance(A_ref, B_ref, A_ref, B_ref),
    "distance_max_error": distance_max_error,
    "samples": distance_rows,
}, ARTIFACT_ROOT, "checks", "distance-from-projective-data.json")

display_artifact(distance_png, width=760)
display_artifact(distance_csv)
distance_max_error

## Applied Lab: What Survives Which Transformation?

Use this lab table as a small diagnostic. Start with one circle, one angle, and one reference segment. Then apply three transformations:

- a Euclidean motion, which should preserve distances and angles;
- a similarity with scale, which should preserve angles and ratios but not absolute distance;
- an affine shear, which is projective and affine but should not preserve the circular pair `{I,J}`.

The table deliberately mixes ordinary Euclidean measurements with the projective tests from the chapter. The learning target is to see the hierarchy from Section 18.4 in numbers rather than just as a taxonomy.

In [ ]:
E = similarity_preserving(1.0, math.radians(-35), 0.2, 0.15)
SIM = similarity_preserving(1.35, math.radians(42), -0.35, 0.10)
SH = shear(0.55)
lab_transforms = [("Euclidean motion", E), ("scaled similarity", SIM), ("affine shear", SH)]

base_circle_points = [A, B, C, D]
base_line_l = line_from_angle(math.radians(12), -0.10)
base_line_m = line_from_angle(math.radians(69), 0.15)
base_angle = laguerre_angle_mod_pi(base_line_l, base_line_m)
base_dist = projective_distance(segments[0][1], segments[0][2], A_ref, B_ref)

lab_rows = []
for name, T in lab_transforms:
    T_inv_T = np.linalg.inv(T).T
    transformed_circle = [T @ p for p in base_circle_points]
    transformed_l = T_inv_T @ base_line_l
    transformed_m = T_inv_T @ base_line_m
    transformed_P = T @ segments[0][1]
    transformed_Q = T @ segments[0][2]
    transformed_A = T @ A_ref
    transformed_B = T @ B_ref
    pair_preserved = ((projectively_equal(T @ I, I) and projectively_equal(T @ J, J)) or (projectively_equal(T @ I, J) and projectively_equal(T @ J, I)))
    angle_after = laguerre_angle_mod_pi(transformed_l, transformed_m)
    abs_dist_after = projective_distance(transformed_P, transformed_Q, A_ref, B_ref)
    ratio_after = projective_distance(transformed_P, transformed_Q, transformed_A, transformed_B)
    lab_rows.append({
        "transformation": name,
        "preserves_circular_pair": bool(pair_preserved),
        "cocircular_residual_after": cocircular_bracket_residual(*transformed_circle),
        "angle_error_mod_pi": mod_pi_error(angle_after, base_angle),
        "distance_against_original_unit": abs_dist_after,
        "distance_ratio_against_transformed_AB": ratio_after,
        "base_distance_against_unit": base_dist,
    })
lab_csv = save_table(lab_rows, ARTIFACT_ROOT, "tables", "projective-measurement-lab.csv")
lab_checks_path = save_json({"rows": lab_rows}, ARTIFACT_ROOT, "checks", "projective-measurement-lab.json")
display_artifact(lab_csv)
lab_rows

## Sanity Checks

The final checks are intentionally redundant. They verify the visual artifacts, the exact symbolic identities, and the numerical invariants that carry the chapter:

- circular points lie on the ideal line and on every circle equation;
- cocircularity has a tiny residual for the circle sample and a visible residual after perturbation;
- robust cross-ratio readings agree;
- similarities preserve or swap the circular pair, while shear does not;
- orthogonality gives cross-ratio `-1`;
- Laguerre's formula recovers angle modulo `pi`;
- the distance formula matches Euclidean distance after unit normalization.

In [ ]:
artifact_paths = [
    storyboard_path,
    core_checks_path,
    circular_points_png,
    circular_points_checks_path,
    cocircular_png,
    cocircular_csv,
    cocircular_checks_path,
    robust_html,
    robust_csv,
    robust_checks_path,
    transform_html,
    transform_checks_path,
    miquel_png,
    miquel_checks_path,
    orthogonality_png,
    orthogonality_checks_path,
    laguerre_png,
    laguerre_html,
    laguerre_csv,
    laguerre_checks_path,
    distance_png,
    distance_csv,
    distance_checks_path,
    lab_csv,
    lab_checks_path,
]

for path in artifact_paths:
    if not Path(path).exists():
        raise AssertionError(f"Missing artifact: {path}")
    if Path(path).stat().st_size <= 128:
        raise AssertionError(f"Artifact too small: {path}")

png_summaries = [png_stats(path) for path in [circular_points_png, cocircular_png, miquel_png, orthogonality_png, laguerre_png, distance_png]]
for summary in png_summaries:
    assert summary["width"] >= 200 and summary["height"] >= 150
    assert summary["pixel_std"] > 1.0

assert core_checks["I_on_line_at_infinity"] and core_checks["J_on_line_at_infinity"]
assert core_checks["I_cross_J_spans_l_infinity"]
assert cocircular_good_residual < 1e-9
assert cocircular_bad_residual > 1e-4
assert cross_ratio_error < 1e-9
assert any(row["preserves_pair"] for row in transformation_checks if "preserving" in row["transformation"])
assert any(row["preserves_pair"] for row in transformation_checks if "reversing" in row["transformation"])
assert not [row for row in transformation_checks if row["transformation"] == "affine shear"][0]["preserves_pair"]
assert orthogonality_residual < 1e-9
assert laguerre_max_error < 1e-12
assert distance_max_error < 1e-12
assert miquel_data["has_path_to_conclusion"]

visual_checks = {
    "all_files_exist": all(Path(path).exists() for path in artifact_paths),
    "cross_ratio_error": cross_ratio_error,
    "numeric_checks": {
        "cocircular_good_residual": cocircular_good_residual,
        "cocircular_bad_residual": cocircular_bad_residual,
        "orthogonality_residual": orthogonality_residual,
        "laguerre_max_error": laguerre_max_error,
        "distance_max_error": distance_max_error,
        "miquel_has_path_to_conclusion": bool(miquel_data["has_path_to_conclusion"]),
    },
    "artifacts": [book_relative(path) for path in artifact_paths],
    "png_summaries": png_summaries,
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final_sanity = {
    "chapter": 18,
    "title": "Euclidean Geometry",
    "source_span": CHAPTER["source_span"],
    "storyboard_items_implemented": [item["artifact"] for item in STORYBOARD["visual_sequence"]],
    "libraries_used": {
        "numpy": "homogeneous coordinates, complex circular points, determinants, transformations",
        "sympy": "exact circular-point and bracket-transfer identities",
        "matplotlib": "durable construction and proof-state PNG artifacts",
        "plotly": "interactive transformation and Laguerre/cross-ratio HTML artifacts",
        "networkx": "Miquel theorem translation dependency graph",
        "pandas": "CSV invariant and lab tables",
    },
    "checks": visual_checks["numeric_checks"],
    "artifact_sizes": artifact_size_map(artifact_paths + [visual_checks_path]),
    "known_gaps": [
        "I and J are complex ideal points, so figures show them schematically on the ideal line rather than as real affine points.",
        "The Miquel section visualizes the projective proof dependencies, not a full synthetic construction of all eight points.",
    ],
}
final_sanity_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")

assert_artifacts([visual_checks_path, final_sanity_path], min_size=256)
{"visual_checks": book_relative(visual_checks_path), "final_sanity": book_relative(final_sanity_path)}

## Takeaways

- The circular points `I` and `J` are the projective memory of Euclidean structure in the standard embedding.
- A circle is a conic through `I` and `J`; cocircularity can therefore be tested as a six-point conic condition.
- Similarities are exactly the real projective maps that preserve the unordered pair `{I,J}`.
- Orthogonality is the harmonic special case of a broader angle formula.
- Laguerre's formula turns an angle between finite lines into a cross-ratio of their ideal directions with `I` and `J`.
- Absolute distance needs one extra datum: a reference segment. Once that unit is fixed, the distance formula is again written entirely in projective brackets.